# Probability Concepts with Real-World Examples

## Real-World Applications of Probability

### Autocorrect Systems
- **How it works**:
  - When you type "recieve", it suggests "receive"
  - Uses word frequency statistics (unigram/bigram probabilities)


### Spam Filters
- **How it works**:
  - Flags messages like "Win 1M!" as spam
  - Uses Bayesian probability to update beliefs


## Probability Terminology Explained

### Key Concepts with Email Examples

#### Random Experiment
- **Definition**: A process that produces an outcome
- **Email example**: Classifying an email as spam/ham
- **Other examples**: Rolling a die, flipping a coin

#### Trial
- **Definition**: One execution of a random experiment
- **Email example**: Processing one email message
- **Other examples**: One coin flip, one die roll

#### Outcome
- **Definition**: The result of a trial
- **Email example**: "spam" or "not spam" classification
- **Other examples**: "heads", "tails", die face number

#### Sample Space
- **Definition**: All possible outcomes
- **Email example**: All possible email texts and classifications
- **Notation**: S = {"spam", "not spam"}


### Corpus in Natural Language Processing

#### Definition
A **corpus** (plural: corpora) is a large, structured collection of texts used for linguistic analysis and training language models.

# Exercise 1: Next-Word Prediction



## Build Your Own Predictor

### Challenge
Design an algorithm that predicts the next word given typing history.

**Example Input/Output:**
- Input: "I want to"
- Output:
  - "go" (45% probability)
  - "eat" (30% probability)
  - "sleep" (25% probability)

## Next-Word Prediction Pipeline

### Step-by-Step Process
1. **Analyze current phrase**  
   Example: "How are"

2. **Generate likely candidates**  
   Potential next words: "you", "doing", "we"

3. **Rank by conditional probability**  
   Calculate P(word | context) for each candidate


# Conceptual implementation
context = "How are"
candidates = ["you", "doing", "we"]

# Calculate probabilities

Probabilities

    "you": 0.72,    # P("you" | "How are")

    "doing": 0.18,   # P("doing" | "How are")

    "we": 0.10       # P("we" | "How are")


# N-gram Language Models

## Markov Assumption
The probability of a word depends only on the previous **k** words:

P(w_n | w_1...w_{n-1}) ≈ P(w_n | w_{n-k}...w_{n-1})


## Unigram Model
**Assumption**: Words are completely independent  
**Probability**:
P(w) = count(w) / total_words

**Example**: "I love you"

P("I love you") = P("I") × P("love") × P("you")


## Bigram Model
**Assumption**: Word depends on 1 previous word  
**Probability**:
P(w_n | w_{n-1}) = count(w_{n-1} w_n) / count(w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"love")

## Trigram Model
**Assumption**: Word depends on 2 previous words  
**Probability**:
P(w_n | w_{n-2}, w_{n-1}) = count(w_{n-2} w_{n-1} w_n) / count(w_{n-2} w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"I love")

## Comparison Table

| Model    | Dependency          | Example Probability            |
|----------|---------------------|--------------------------------|
| Unigram  | None                | P("love")                     |
| Bigram   | 1 previous word     | P("you"/"love")              |
| Trigram  | 2 previous words    | P("you"/"I love")            |





### Dataset
Using Shakespeare's sonnets (public domain):
```python
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [7]:
!pip install wget

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9711 sha256=e96cb4afa3d399a8a7348370d8054ccc1dc00b45605ef653094fe6286a3b2e7d
  Stored in directory: c:\users\art\appdata\local\pip\cache\wheels\8a\b8\04\0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


In [12]:
import wget
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
wget.download(url)

'input.txt'

#Data Loading & Preprocessing

In [13]:
import re

def load_data(filepath):
    """Load text file and return as string"""
    try:
        with open(filepath, 'r', encoding='utf-8') as file:
            text = file.read()
        return text
    except FileNotFoundError:
        print("Error: File not found.")
        return ""

def preprocess(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing special characters
    3. Splitting into tokens (words)
    """

    # Convert to lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r"[^a-z0-9'\s]", "", text)

    # Split into tokens (words)
    tokens = text.split()

    return tokens


# TEST
text = load_data("input.txt")

if text:
    tokens = preprocess(text)
    print("First 50 tokens:", tokens[:50])

First 50 tokens: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen', 'first', 'you', 'know', 'caius', 'marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', 'all', 'we', "know't", 'we', "know't", 'first', 'citizen', 'let', 'us']


# Model Implementation

In [14]:
import re
from collections import defaultdict

def load_data(filepath):
    """Load text file and return as string"""
    try:
        with open(filepath, 'r', encoding='utf-8') as file:
            text = file.read()
        return text
    except FileNotFoundError:
        print("Error: File not found.")
        return None


def preprocess(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing special characters
    3. Splitting into tokens
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", "", text)
    tokens = text.split()
    return tokens


def build_ngram_model(tokens, n=2):
    """
    Build n-gram model (unigram, bigram, trigram)
    Example for bigram:
    {"i": {"am": 5, "have": 3}}
    """

    model = defaultdict(lambda: defaultdict(int))

    # sliding window
    for gram in zip(*[tokens[i:] for i in range(n)]):

        context = gram[:-1]
        next_word = gram[-1]

        if n == 2:
            context = context[0]  # make bigram cleaner

        model[context][next_word] += 1

    return model


def predict_next_word(model, context, k=3):
    """
    Predict next words given context
    Returns: list of (word, probability)
    """

    if context not in model:
        return []

    next_words = model[context]
    total = sum(next_words.values())

    probs = []

    for word, count in next_words.items():
        probability = count / total
        probs.append((word, probability))

    probs = sorted(probs, key=lambda x: x[1], reverse=True)

    return probs[:k]


# =========================
# TEST THE MODEL
# =========================

text = load_data("input.txt")

if text:

    tokens = preprocess(text)

    # Build Bigram Model
    bigram_model = build_ngram_model(tokens, n=2)

    print("Sample context: 'i'")
    predictions = predict_next_word(bigram_model, "i", k=5)

    print("Top predictions:")
    for word, prob in predictions:
        print(word, "->", round(prob, 3))

Sample context: 'i'
Top predictions:
am -> 0.082
have -> 0.079
will -> 0.062
do -> 0.037
would -> 0.032


#Testing

In [15]:
bigram_model = build_ngram_model(tokens, n=2)
print("Words after 'the':", predict_next_word(bigram_model, "the"))

Words after 'the': [('king', 0.024685459468068164), ('duke', 0.0168816690555821), ('world', 0.015766841853798376)]


# Exercise 2: Spam/Ham Classification



## Spam vs Ham Examples

<div style="display: flex;">
<div style="flex: 50%; padding: 10px;">

### Spam Examples
- "Claim your free \$1M"
- "You won an iPhone!"
- "Limited time offer!"
- "Click here to claim your prize"

</div>
<div style="flex: 50%; padding: 10px;">

### Ham Examples
- "Meeting at 3 PM"
- "Project update attached"
- "Let's discuss the proposal"
- "Quarterly report is ready"

</div>
</div>


### 1. Prior Probability (P(c))
**What it represents:**  
The baseline probability of a class before examining the message content.

**Example Calculation:**  

#### In 100 emails:
spam_count = 30

ham_count = 70

P_spam = spam_count/100  # 0.30

P_ham = ham_count/100    # 0.70

### 2. Likelihood (P(w|c))

**Definition:**  
The probability of observing a specific word given a particular class (spam/ham). This measures how characteristic a word is for a certain class.


#### Spam class: 25 occurrences out of 1000 total words
P_free_spam = 25/1000  # 0.025

#### Ham class: 2 occurrences out of 3000 total words
P_free_ham = 2/3000    # ≈0.00067

### 3. Posterior Probability

**Definition:**
The posterior probability **P(c|msg)** represents:
- The probability that a given message belongs to class *c* (spam/ham)
- The key quantity we compute to make classification decisions

### Bayesian Classification

### Decision Rule
The classifier selects the class with the highest posterior probability:

Where:
- `P(c|msg)` = Posterior probability (what we want to calculate)
- `P(msg|c)` = Likelihood of the message given the class
- `P(c)` = Prior probability of the class

### Practical Example: "free prize" message

**Given Probabilities:**

### Word probabilities
P_free_spam = 0.025

P_free_ham = 0.00067

P_prize_spam = 0.015

P_prize_ham = 0.001



### Class priors
P_spam = 0.3

P_ham = 0.7

### Calculation Steps
#### Compute Spam Probability:
P_msg_spam = P_free_spam * P_prize_spam * P_spam
           = 0.025 × 0.015 × 0.3
           ≈ 0.0001125

####Compute Ham Probability:

P_msg_ham = P_free_ham * P_prize_ham * P_ham
          = 0.00067 × 0.001 × 0.7
          ≈ 0.000000469

####Comparison:
0.0001125 (Spam) > 0.000000469 (Ham)

##Laplace Smoothing
**Problem**:  
P("xyz"|Spam) = 0 if "xyz" never appeared in training spam

**Solution**:  
$$ P_{\text{smooth}}(w|c) = \frac{\text{count}(w,c) + 1}{\text{count}(c) + V} $$

*Example*:  
- V = 10,000 words  
- count("prize", Spam) = 0  
- count(Spam) = 1000  
$$ P_{\text{smooth}}("prize"|Spam) = \frac{0+1}{1000+10000} \approx 0.00009 $$



## Stop Words in Text Processing

**Definition:** Common words (e.g., "the", "a", "and") that appear frequently but carry minimal meaning.

**Why Remove Them?**
1. **Meaning Preservation:**  
   "The quick brown fox" → "quick brown fox" (keeps core meaning)
   
2. **Improved Analysis:**  
   - Without: P("the"|spam)=0.101 vs P("the"|ham)=0.098 (useless)  
   - With: P("free"|spam)=0.025 vs P("free"|ham)=0.00067 (discriminative)

3. **Efficiency:**  
   Reduces vocabulary size by 30-40% (1000 words → 600 words)

**Example: Spam Detection**
```python
from sklearn.feature_extraction.text import CountVectorizer

### With stop words
text = ["You have won a free prize"]
vectorizer = CountVectorizer()
print("With stop words:", vectorizer.fit_transform(text).toarray())
### Output: [[1 1 1 1 1]] (for "you", "have", "won", "free", "prize")

### Without stop words
vectorizer = CountVectorizer(stop_words='english')
print("Without stop words:", vectorizer.fit_transform(text).toarray())
### Output: [[1 1 1]] (for "won", "free", "prize")

## Dataset Loading

In [18]:
import requests
import zipfile
import io

# 1. Download the file
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
response = requests.get(url)

# 2. Extract the zip content
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall()

print("File downloaded and extracted successfully!")

File downloaded and extracted successfully!


In [20]:
import re

def load_sms_data(filepath):
    messages = []
    labels = []
    # Using 'utf-8' is good, but sometimes this specific file likes 'latin-1'
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    labels.append(parts[0])
                    messages.append(parts[1])
    except FileNotFoundError:
        print(f"Error: The file '{filepath}' was not found. Did you unzip it?")
    return messages, labels

def preprocess_text(text, remove_stopwords=True):
    # Basic cleaning
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()

    if remove_stopwords:
        stop_words = {
            'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves',
            'you', 'your', 'yours', 'yourself', 'yourselves', 'he', 'him',
            'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its',
            'itself', 'they', 'them', 'their', 'theirs', 'themselves',
            'what', 'which', 'who', 'whom', 'this', 'that', 'these',
            'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
            'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
            'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as',
            'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
            'against', 'between', 'into', 'through', 'during', 'before',
            'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in',
            'out', 'on', 'off', 'over', 'under', 'again', 'further',
            'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
            'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other',
            'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same',
            'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just',
            'don', 'should', 'now'
        }
        words = text.split()
        text = ' '.join([w for w in words if w not in stop_words])

    return text

# --- EXECUTION ---
# Make sure the filename matches what was unzipped!
messages, labels = load_sms_data("SMSSpamCollection")

if messages:
    print(f"Successfully loaded {len(messages)} messages.")
    print("-" * 30)
    print("Original: ", messages[0])
    print("Cleaned:  ", preprocess_text(messages[0], remove_stopwords=True))

Successfully loaded 5574 messages.
------------------------------
Original:  Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Cleaned:   go jurong point crazy available bugis n great world la e buffet cine got amore wat


#Calculate Prior Probabilities

In [24]:
def calculate_priors(labels):
    """
    Compute P(spam) and P(ham) from the list of labels.
    Returns: A dictionary with 'spam' and 'ham' probabilities.
    """
    total_messages = len(labels)
    
    # Count occurrences of each label
    spam_count = labels.count('spam')
    ham_count = labels.count('ham')
    
    # Calculate probabilities
    p_spam = spam_count / total_messages
    p_ham = ham_count / total_messages
    
    # Return as a dictionary for easy access later
    prior_prob = {
        'spam': p_spam,
        'ham': p_ham
    }
    
    return prior_prob

# --- Testing the function ---
# Assuming 'labels' is the list from your previous code
priors = calculate_priors(labels)

print(f"Total Messages: {len(labels)}")
print(f"P(Spam): {priors['spam']:.4f}")
print(f"P(Ham):  {priors['ham']:.4f}")

Total Messages: 5574
P(Spam): 0.1340
P(Ham):  0.8660


#Count Words per Class

In [25]:
def count_words(messages, labels):
    """
    Counts how many times each word appears in 'spam' vs 'ham' messages.
    """
    # Initialize counts for both classes
    word_counts = {
        'spam': {},
        'ham': {}
    }
    vocab = set()

    # Iterate through each message and its corresponding label
    for message, label in zip(messages, labels):
        # 1. Clean the message using the function we wrote earlier
        cleaned_text = preprocess_text(message, remove_stopwords=True)
        words = cleaned_text.split()
        
        # 2. Update counts for the specific class
        for word in words:
            # Add to the vocabulary set
            vocab.add(word)
            
            # Update the frequency in the nested dictionary
            if word not in word_counts[label]:
                word_counts[label][word] = 1
            else:
                word_counts[label][word] += 1
                
    return word_counts, vocab

# --- Testing the function ---
word_counts, vocab = count_words(messages, labels)

print(f"Total Unique Words (Vocab Size): {len(vocab)}")
print(f"Top 5 'spam' words: {dict(list(word_counts['spam'].items())[:5])}")

Total Unique Words (Vocab Size): 9464
Top 5 'spam' words: {'free': 216, 'entry': 26, '2': 173, 'wkly': 14, 'comp': 10}


#Compute Word Probabilities (with Smoothing)

In [26]:
def calculate_word_probs(word_counts, vocab, k=1):
    """
    Calculate P(word|spam) and P(word|ham) for every word in the vocabulary.
    Using Laplace Smoothing (k).
    """
    word_probs = {
        'spam': {},
        'ham': {}
    }
    
    # 1. Calculate the total number of words in each category
    # (This is the sum of all frequencies in each dictionary)
    total_spam_words = sum(word_counts['spam'].values())
    total_ham_words = sum(word_counts['ham'].values())
    
    # 2. Get the Vocabulary size (V)
    V = len(vocab)
    
    # 3. For every word in our known universe (vocab)...
    for word in vocab:
        # Calculate P(word | spam)
        # Formula: (Count in spam + k) / (Total words in spam + k * V)
        spam_word_count = word_counts['spam'].get(word, 0)
        word_probs['spam'][word] = (spam_word_count + k) / (total_spam_words + k * V)
        
        # Calculate P(word | ham)
        ham_word_count = word_counts['ham'].get(word, 0)
        word_probs['ham'][word] = (ham_word_count + k) / (total_ham_words + k * V)
        
    return word_probs

# --- Execution ---
word_probs = calculate_word_probs(word_counts, vocab)

# Let's peek at a common spam word like 'free' or 'win' if they exist
test_word = 'free'
if test_word in vocab:
    print(f"P('{test_word}' | Spam): {word_probs['spam'][test_word]:.6f}")
    print(f"P('{test_word}' | Ham):  {word_probs['ham'][test_word]:.6f}")

P('free' | Spam): 0.009818
P('free' | Ham):  0.001205


#Predict Spam Probability

In [27]:
import math

def predict(message, prior_prob, word_probs, vocab):
    """
    Predicts the probability that a message is spam.
    """
    # 1. Preprocess message and split into words
    cleaned_message = preprocess_text(message, remove_stopwords=True)
    words = cleaned_message.split()

    # 2. Initialize log probabilities with the Prior (starting belief)
    # Using log(P(spam)) and log(P(ham))
    log_prob_spam = math.log(prior_prob['spam'])
    log_prob_ham = math.log(prior_prob['ham'])

    # 3. For each word, add its log probability to the running total
    for word in words:
        if word in vocab:
            log_prob_spam += math.log(word_probs['spam'][word])
            log_prob_ham += math.log(word_probs['ham'][word])
            
    # 4. Convert log probabilities back to normal scale using exp()
    # We use these relative scores to find the final probability
    spam_score = math.exp(log_prob_spam)
    ham_score = math.exp(log_prob_ham)

    # 5. Return P(spam|message) = spam_score / (spam_score + ham_score)
    # This normalizes the result between 0 and 1
    probability = spam_score / (spam_score + ham_score)
    
    return probability

# --- Testing the Predictor ---
test_msg = "WINNER! You have won a 1000 cash prize. Call now to claim your FREE gift!"
spam_chance = predict(test_msg, priors, word_probs, vocab)

print(f"Message: {test_msg}")
print(f"Probability of being SPAM: {spam_chance:.4%}")

Message: WINNER! You have won a 1000 cash prize. Call now to claim your FREE gift!
Probability of being SPAM: 100.0000%


In [28]:
# 1. Test calculate_priors()
# To get {'spam': 0.33, 'ham': 0.67}, we need 1 spam and 2 ham messages
test_labels = ['spam', 'ham', 'ham'] 
priors = calculate_priors(test_labels)
print(f"Priors: {priors}") 

# 2. Test count_words()
# We match the messages to the labels (1 spam, 1 ham)
test_messages = ["free prize", "meeting today"]
test_labels_2 = ['spam', 'ham']
word_counts, vocab = count_words(test_messages, test_labels_2)

print(f"Count of 'free' in spam: {word_counts['spam'].get('free', 0)}") 
print(f"Vocab size: {len(vocab)}") # 'free', 'prize', 'meeting', 'today' = 4

# 3. Test calculate_word_probs()
# Using the word_counts and vocab from the step above
word_probs = calculate_word_probs(word_counts, vocab, k=1)

# Manual Math check:
# Count of 'free' in spam = 1
# k = 1
# Total words in spam = 2 ("free", "prize")
# V = 4
# Prob = (1 + 1) / (2 + 1 * 4) = 2 / 6 = 0.333
print(f"P('free'|spam): {word_probs['spam'].get('free', 0):.2f}")

Priors: {'spam': 0.3333333333333333, 'ham': 0.6666666666666666}
Count of 'free' in spam: 1
Vocab size: 4
P('free'|spam): 0.33
